# 1. Imports

In [ ]:
import csv
import json
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import numpy as np
import optuna
import seaborn as sns
import torch
from mlflow.tracking import MlflowClient
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from ultralytics import YOLO

# 2. Configuration

In [ ]:
sys.path.append("../src")
from preprocessing import CLASSES, NUM_CLASSES, WasteDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
print(f"Classes : {NUM_CLASSES} — {CLASSES}")

# 3. MLflow + load data

In [ ]:
MLFLOW_URI = "http://localhost:5000"
EXPERIMENT_NAME = "sortify-yolo"
REGISTERED_MODEL = "sortify-yolo-cls"
SPLITS_PATH = "../data/splits/splits.json"
SAVED_DIR = Path("../saved")
YOLO_DATA_DIR = Path("../data/yolo")
SAVED_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

class_weights = (
    WasteDataset(splits["train"]).get_class_weights().to(device)
)

print(
    f"Train : {len(splits['train'])} | "
    f"Val : {len(splits['val'])} | "
    f"Test : {len(splits['test'])}"
)
print(
    f"Class weights : "
    f"{[round(w, 3) for w in class_weights.tolist()]}"
)

# 4. Preparing the dataset for YOLO Classification
YOLO classification is waiting for the structure:
```
yolo/
  train/
    glass/img1.jpg ...
    metal/img2.jpg ...
  val/
    glass/...
  test/
    glass/...
```

In [ ]:
def build_yolo_dataset(
    splits: dict,
    dst: Path,
) -> None:
    """Copy the images into the format expected by YOLO."""
    if dst.exists():
        shutil.rmtree(dst)

    for split_name, samples in splits.items():
        for sample in samples:
            src = Path(sample["path"])
            label = sample["label"]
            target = dst / split_name / label / src.name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, target)

    print(f"Dataset YOLO construit dans : {dst}")
    for split_name in splits:
        n = sum(1 for _ in (dst / split_name).rglob("*.jpg"))
        print(f"  {split_name:<6} : {n} images")


build_yolo_dataset(splits, YOLO_DATA_DIR)

# 5. Optuna — Hyperparameter Search
YOLO Classification uses its own trainer.
Optuna is looking for : lr0, dropout, batch, imgsz, modele (nano/small).

In [ ]:
N_TRIALS = 20
EPOCHS_OPTUNA = 8


def objective(trial: optuna.Trial) -> float:
    lr0 = trial.suggest_float("lr0", 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    batch = trial.suggest_categorical("batch", [16, 32, 64])
    imgsz = trial.suggest_categorical("imgsz", [224, 320])
    model_size = trial.suggest_categorical(
        "model_size", ["yolov8n-cls", "yolov8s-cls"]
    )
    weight_decay = trial.suggest_float(
        "weight_decay", 1e-5, 1e-3, log=True
    )

    run_dir = SAVED_DIR / f"yolo_trial_{trial.number}"

    model = YOLO(f"{model_size}.pt")
    results = model.train(
        data=str(YOLO_DATA_DIR),
        epochs=EPOCHS_OPTUNA,
        imgsz=imgsz,
        batch=batch,
        lr0=lr0,
        dropout=dropout,
        weight_decay=weight_decay,
        project=str(run_dir),
        name="train",
        exist_ok=True,
        verbose=False,
        device=device,
    )

    val_acc = float(results.results_dict.get("metrics/accuracy_top1", 0))
    val_loss = 1.0 - val_acc

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        mlflow.log_params({
            "lr0": lr0,
            "dropout": dropout,
            "batch": batch,
            "imgsz": imgsz,
            "model_size": model_size,
            "weight_decay": weight_decay,
            "phase": "optuna",
        })
        mlflow.log_metrics({
            "val_acc": val_acc,
            "val_loss": val_loss,
        })

    # Clean up to avoid filling up the disk
    shutil.rmtree(run_dir, ignore_errors=True)

    trial.report(val_loss, EPOCHS_OPTUNA)
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

    return val_loss


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5, n_warmup_steps=3
    ),
    study_name="sortify-yolo-optuna",
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print("\nMeilleurs hyperparametres :")
for k, v in best.items():
    print(f"  {k:<15} : {v}")
print(f"Meilleur val_loss : {study.best_value:.4f}")

In [ ]:
# Optuna Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trial_values = [
    t.value for t in study.trials if t.value is not None
]
axes[0].plot(
    trial_values,
    marker="o",
    color="#7F77DD",
    markersize=5,
    linewidth=1,
)
axes[0].axhline(
    study.best_value,
    color="#D85A30",
    linestyle="--",
    label=f"Best: {study.best_value:.4f}",
)
axes[0].set_title("Val loss par trial")
axes[0].set_xlabel("Trial")
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

importances = optuna.importance.get_param_importances(study)
axes[1].barh(
    list(importances.keys()),
    list(importances.values()),
    color="#5DCAA5",
)
axes[1].set_title("Importance des hyperparametres")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(SAVED_DIR / "yolo_optuna.png", dpi=150)
plt.show()

# 6. Final training with the best hyperparameters

In [ ]:
EPOCHS_FINAL = 50
FINAL_DIR = SAVED_DIR / "yolo_final"

model = YOLO(f"{best['model_size']}.pt")

with mlflow.start_run(run_name="final_training") as final_run:
    mlflow.log_params({
        **best,
        "epochs": EPOCHS_FINAL,
        "architecture": "YOLOv8-cls",
        "phase": "final",
    })

    results = model.train(
        data=str(YOLO_DATA_DIR),
        epochs=EPOCHS_FINAL,
        imgsz=best["imgsz"],
        batch=best["batch"],
        lr0=best["lr0"],
        dropout=best["dropout"],
        weight_decay=best["weight_decay"],
        project=str(FINAL_DIR),
        name="train",
        exist_ok=True,
        verbose=True,
        device=device,
    )

    # Metrics logs, epoch by epoch, from the YOLO CSV files
    csv_path = FINAL_DIR / "train" / "results.csv"
    if csv_path.exists():
        import csv

        with open(csv_path) as f:
            rows = list(csv.DictReader(f))
        for i, row in enumerate(rows, 1):
            metrics = {
                k.strip(): float(v)
                for k, v in row.items()
                if v.strip()
            }
            mlflow.log_metrics(metrics, step=i)

    # Evaluation on the test set
    best_weights = FINAL_DIR / "train" / "weights" / "best.pt"
    model_eval = YOLO(str(best_weights))
    test_results = model_eval.val(
        data=str(YOLO_DATA_DIR),
        split="test",
        device=device,
        verbose=False,
    )

    test_acc = float(
        test_results.results_dict.get("metrics/accuracy_top1", 0)
    )

    # Manual predictions for confusion matrix and F1 scores
    all_preds, all_labels = [], []
    for sample in splits["test"]:
        pred = model_eval(
            sample["path"], verbose=False
        )[0].probs.top1
        all_preds.append(pred)
        all_labels.append(CLASSES.index(sample["label"]))

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    test_f1 = float(
        f1_score(all_labels, all_preds, average="macro")
    )

    mlflow.log_metrics({
        "test_acc": test_acc,
        "test_f1_macro": test_f1,
    })

    # Log the YOLO model as an artifact
    mlflow.log_artifact(str(best_weights), artifact_path="model")

    print(f"Test accuracy  : {test_acc:.4f}")
    print(f"Test F1 macro  : {test_f1:.4f}")
    print(f"Run ID         : {final_run.info.run_id}")

# 7. Learning curves from YOLO CSV files

In [ ]:
csv_path = FINAL_DIR / "train" / "results.csv"
epochs_list, train_loss_list, val_acc_list = [], [], []

with open(csv_path) as f:
    for i, row in enumerate(csv.DictReader(f), 1):
        row = {k.strip(): v.strip() for k, v in row.items()}
        epochs_list.append(i)
        train_loss_list.append(
            float(row.get("train/loss", row.get("loss", 0)))
        )
        val_acc_list.append(
            float(row.get("metrics/accuracy_top1", 0))
        )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    epochs_list, train_loss_list, color="#5DCAA5", linewidth=1.5
)
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].plot(
    epochs_list, val_acc_list, color="#D85A30", linewidth=1.5
)
axes[1].set_title("Val Accuracy Top-1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle(
    "YOLOv8 Classification — Courbes d'apprentissage",
    fontsize=13,
)
plt.tight_layout()
plt.savefig(SAVED_DIR / "yolo_learning_curves.png", dpi=150)
plt.show()

# 8. Confusion matrix + classification report

In [ ]:
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=CLASSES,
        digits=3,
    )
)

cm_norm = confusion_matrix(all_labels, all_preds).astype(float)
cm_norm /= cm_norm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    cmap="YlOrRd",
    ax=ax,
    linewidths=0.5,
)
ax.set_xlabel("Predit", fontsize=12)
ax.set_ylabel("Reel", fontsize=12)
ax.set_title(
    "Matrice de confusion normalisee — YOLOv8",
    fontsize=13,
    pad=15,
)
plt.tight_layout()
plt.savefig(SAVED_DIR / "yolo_confusion_matrix.png", dpi=150)
plt.show()

# 9. Comparison of the 3 models + Model Registry promotion

In [ ]:
client = MlflowClient(MLFLOW_URI)

# Load stats from previous models
all_stats = [{"model": "YOLOv8", "test_accuracy": test_acc, "test_f1_macro": test_f1}]

for name, path in [
    ("CNN Scratch", SAVED_DIR / "scratch_stats.json"),
    ("MobileNetV2", SAVED_DIR / "mobilenet_stats.json"),
]:
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        all_stats.append({
            "model": name,
            "test_accuracy": data["test_accuracy"],
            "test_f1_macro": data["test_f1_macro"],
        })

# Comparison table
print(f"  {'Modele':<15} {'Test Acc':>10} {'Test F1':>10}")
print(f"  {'-' * 38}")
best_model = max(all_stats, key=lambda x: x["test_accuracy"])
for s in all_stats:
    flag = " <-- meilleur" if s["model"] == best_model["model"] else ""
    print(
        f"  {s['model']:<15} "
        f"{s['test_accuracy']:>10.4f} "
        f"{s['test_f1_macro']:>10.4f}"
        f"{flag}"
    )

# Save YOLO stats
with open(SAVED_DIR / "yolo_stats.json", "w") as f:
    json.dump(
        {
            "model": "yolov8_cls",
            "test_accuracy": round(test_acc, 4),
            "test_f1_macro": round(test_f1, 4),
            "hyperparams": best,
        },
        f,
        indent=2,
    )

# Comparison visualization
fig, ax = plt.subplots(figsize=(9, 5))
models = [s["model"] for s in all_stats]
accs = [s["test_accuracy"] for s in all_stats]
f1s = [s["test_f1_macro"] for s in all_stats]
x = np.arange(len(models))
width = 0.35

ax.bar(x - width / 2, accs, width, label="Test Accuracy", color="#5DCAA5")
ax.bar(x + width / 2, f1s, width, label="Test F1 macro", color="#7F77DD")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1)
ax.set_title("Comparaison des 3 modeles Sortify", fontsize=13)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(SAVED_DIR / "models_comparison.png", dpi=150)
plt.show()

# Promote to staging if test_acc > 0.85
try:
    versions = client.get_latest_versions(REGISTERED_MODEL)
    last_ver = versions[-1].version
    if test_acc >= 0.85:
        client.transition_model_version_stage(
            name=REGISTERED_MODEL,
            version=last_ver,
            stage="Staging",
        )
        print(f"\nYOLO v{last_ver} promu en Staging (test_acc={test_acc:.4f})")
    else:
        print(
            f"\nYOLO v{last_ver} reste en None "
            f"— test_acc={test_acc:.4f} < 0.85"
        )
except Exception as e:
    print(f"Registry non disponible : {e}")